<a href="https://colab.research.google.com/github/znr6s28ynr-maker/Deep-Learning-with-Python-Experimentation/blob/main/MNIST_Example_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import keras
from keras import ops

# class creates a matrix W of shape (input_size, output_size), initialized with random variables drawn from a uniform distribution
class NaiveDense:
    def __init__(self, input_size, output_size, activation=None):
      self.activation = activation

    #creates vector b of shape (output_size,), intialized with zeros
      self.W = keras.Variable(
        shape=(input_size, output_size,), initializer="uniform"
      )
      self.b = keras.Variable(shape=(output_size,), initializer="zeros")

    #applies forward pass
    def __call__(self, inputs):
      x = ops.matmul(inputs, self.W)
      x = x + self.b
      if self.activation is not None:
        x = self.activation(x)
      return x

    #convenience method for retrieving the layer's weights
    @property
    def weights(self):
      return(self.W, self.b)

In [23]:
# class chains the layers

class NaiveSequential:
  def __init__(self, layers):
    self.layers = layers

  def __call__(self, inputs):
    x = inputs
    for layer in self.layers:
      x = layer(x)
    return x\

  @property
  def weights(self):
    weights = []
    for layer in self.layers:
      weights += layer.weights
    return weights

In [24]:
#creating mock keras model using NaiveDense and NaiveSequential class
model = NaiveSequential(
    [
        NaiveDense(input_size = 28 * 28, output_size = 512, activation=ops.relu),
        NaiveDense(input_size = 512, output_size = 10, activation=ops.softmax),
    ]
)
assert len(model.weights) == 4

In [25]:
from jax._src.sharding_impls import local_to_global_shape
#iterating over the MNIST data in mini-batches of 128
import math

class BatchGenerator:
  def __init__(self, images, labels, batch_size=128):
    assert len(images) == len(labels)
    self.index = 0
    self.images = images
    self.labels = labels
    self.batch_size = batch_size
    self.num_batches = math.ceil(len(images) / batch_size)

  def next(self):
    images = self.images[self.index : self.index + self.batch_size]
    labels = self.labels[self.index : self.index + self.batch_size]
    self.index += self.batch_size
    return images, labels

In [26]:
# a single step of training

#runs forward pass
def one_training_step(model, images_batch, labels_batch):
  with tf.GradientTape() as tape:
    predictions = model(images_batch)
    loss = ops.sparse_categorical_crossentropy(labels_batch, predictions)
    average_loss = ops.mean(loss)
  gradients = tape.gradient(average_loss, model.weights)
  update_weights(gradients, model.weights)
  return average_loss

In [27]:
learning_rate = 1e-3

def update_weights(gradients, weights):
  for g, w, in zip(gradients, weights):

    #assigns new value to variable
    w.assign(w - g * learning_rate)

In [28]:
import tensorflow as tf

x = tf.zeros(shape=())

#opens GradientTape scope
with tf.GradientTape() as tape:

  #inside scope, applies tensor operations to variable
  y = 2 * x + 3

#uses tape to retrieve the gradient of the output y with respect to x
grad_of_y_wrt_x = tape.gradient(y, x)

In [29]:
#fitting data by implemention of entire epoch of training

def fit(model, images, labels, epochs, batch_size = 128):
  for epoch_counter in range(epochs):
    print(f"Epoch {epoch_counter}")
    batch_generator = BatchGenerator(images, labels)
    for batch_counter in range(batch_generator.num_batches):
      images_batch, labels_batch = batch_generator.next()
      loss = one_training_step(model, images_batch, labels_batch)
      if batch_counter % 100 == 0:
        print(f"loss at batch {batch_counter}: {loss:.2f}")

In [30]:
#testing it

from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

In [31]:
#training model

fit(model, train_images, train_labels, epochs=10, batch_size=128)

Epoch 0
loss at batch 0: 2.30
loss at batch 100: 2.28
loss at batch 200: 2.21
loss at batch 300: 2.19
loss at batch 400: 2.14
Epoch 1
loss at batch 0: 2.12
loss at batch 100: 2.12
loss at batch 200: 2.03
loss at batch 300: 2.02
loss at batch 400: 1.97
Epoch 2
loss at batch 0: 1.94
loss at batch 100: 1.96
loss at batch 200: 1.84
loss at batch 300: 1.84
loss at batch 400: 1.78
Epoch 3
loss at batch 0: 1.74
loss at batch 100: 1.78
loss at batch 200: 1.64
loss at batch 300: 1.65
loss at batch 400: 1.60
Epoch 4
loss at batch 0: 1.54
loss at batch 100: 1.60
loss at batch 200: 1.44
loss at batch 300: 1.46
loss at batch 400: 1.43
Epoch 5
loss at batch 0: 1.36
loss at batch 100: 1.42
loss at batch 200: 1.26
loss at batch 300: 1.29
loss at batch 400: 1.27
Epoch 6
loss at batch 0: 1.20
loss at batch 100: 1.26
loss at batch 200: 1.10
loss at batch 300: 1.14
loss at batch 400: 1.15
Epoch 7
loss at batch 0: 1.06
loss at batch 100: 1.13
loss at batch 200: 0.97
loss at batch 300: 1.02
loss at batch 40

In [32]:
#evaluating model

predictions = model(test_images)
predicted_labels = ops.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
f"accuracy: {ops.mean(matches):.2f}"

'accuracy: 0.84'